# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Michael Bryan  
**Student ID:** *(fill in before submission)*  
**Group:** *(fill in before submission)*  
**Repository:** https://github.com/MICHAELCUY-lang/css4007-hw3-michaelbryan

## AI tool disclosure

I used OpenAI Codex to help structure the Python pipeline, check the arithmetic in Part A, and debug the notebook. I reviewed the code and based the comparisons in Parts B and C on the outputs saved by this notebook.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.


---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

Lowercasing, splitting on spaces and hyphens, and dropping punctuation gives:

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = (9 + 10 + 6) / 3 = 8.3333**

**Query terms:** how, does, rag, retrieve, information

| Query term | f(t,D1) | f(t,D2) | f(t,D3) | n(t) | idf(t) = ln((3-n+0.5)/(n+0.5)) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

Every query-term frequency is zero. Therefore every numerator contains `f(t,d)=0`:

- D1: `0 + 0 + 0 + 0 + 0 = 0`
- D2: `0 + 0 + 0 + 0 + 0 = 0`
- D3: `0 + 0 + 0 + 0 + 0 = 0`

| Document | BM25 score | Rank |
|---|---:|---:|
| D1 | 0.0000 | tied 1st |
| D2 | 0.0000 | tied 1st |
| D3 | 0.0000 | tied 1st |


## A1 (b) — BM25 with the specified stems

| Doc | Stemmed tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = 8.3333**

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t,D1) | f(t,D2) | f(t,D3) | n(t) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | ln(2.5/1.5) = 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

Only `retriev` contributes. For D2 the length-normalisation term is
`1.5 × (1 - 0.75 + 0.75 × 10/8.3333) = 1.7250`.

- D1: `0`
- D2: `0.5108 × [2 × (1.5+1)] / [2 + 1.7250] = 0.6857`
- D3: `0`

| Document | BM25 score | Rank |
|---|---:|---:|
| D1 | 0.0000 | tied 2nd |
| D2 | 0.6857 | 1st |
| D3 | 0.0000 | tied 2nd |

**Which document comes first, and why?** Without stemming all documents tie at zero, because the literal query token `retrieve` does not equal `retrieval` or `retrieves`. With stemming, both forms become `retriev`, so D2 is correctly ranked first. The zero-score result in (a) is a real lexical-mismatch failure, not an arithmetic error.


## A2 — cosine similarity

`‖Q‖ = sqrt(0.8² + 0.2² + 0.5²) = sqrt(0.93) = 0.9644`.

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Q·D1 | 0.8×0.7 + 0.2×0.1 + 0.5×0.4 = 0.78 | 0.9644 | sqrt(0.66)=0.8124 | 0.9956 |
| Q·D2 | 0.8×0.9 + 0.2×0.3 + 0.5×0.6 = 1.08 | 0.9644 | sqrt(1.26)=1.1225 | 0.9977 |
| Q·D3 | 0.8×0.1 + 0.2×0.8 + 0.5×0.2 = 0.34 | 0.9644 | sqrt(0.69)=0.8307 | 0.4244 |

| Document | Cosine similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Most similar:** D2.

**Comparison with BM25:** It is not the same as unstemmed BM25, which cannot distinguish the documents and gives all three zero. Its first result agrees with stemmed BM25 (D2), although cosine also orders D1 above D3 rather than tying them.

**Why they disagree here:** BM25 needs literal token overlap, so `retrieve` misses `retrieval` and `retrieves` until they are stemmed. The supplied dense vectors already place the query close to D2's retrieval/generation meaning even though the surface word forms differ.


Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
import math, re
import numpy as np
q = np.array([0.8, 0.2, 0.5])
docs_a = {"D1": np.array([0.7, 0.1, 0.4]), "D2": np.array([0.9, 0.3, 0.6]), "D3": np.array([0.1, 0.8, 0.2])}
for name, vec in docs_a.items():
    dot = q @ vec
    cosine = dot / (np.linalg.norm(q) * np.linalg.norm(vec))
    print(name, f"dot={dot:.4f}", f"|Q|={np.linalg.norm(q):.4f}", f"|D|={np.linalg.norm(vec):.4f}", f"cos={cosine:.4f}")


D1 dot=0.7800 |Q|=0.9644 |D|=0.8124 cos=0.9956
D2 dot=1.0800 |Q|=0.9644 |D|=1.1225 cos=0.9977
D3 dot=0.3400 |Q|=0.9644 |D|=0.8307 cos=0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json, os, re
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi

load_dotenv()
corpus = [json.loads(line) for line in Path("data/corpus.jsonl").read_text().splitlines()]
fixed_queries = json.loads(Path("data/queries.json").read_text())["queries"]
qa = json.loads(Path("data/qa.json").read_text())["questions"]
by_id = {d["id"]: d for d in corpus}
print(f"Loaded {len(corpus)} chunks, {len(fixed_queries)} fixed queries, and {len(qa)} Part C questions.")


Loaded 74 chunks, 8 fixed queries, and 6 Part C questions.


## B1 — BM25 with `rank_bm25`

**My tokenisation:** lowercase Unicode word tokens using `re.findall(r"\w+", text.lower())`. I keep stop words and numbers, and use no stemming. This makes exact codes such as `GR-305` become the two tokens `gr` and `305`.


In [3]:
def tokenize(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

tokenized_corpus = [tokenize(d["text"]) for d in corpus]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def bm25_retrieve(text, k=3):
    scores = bm25.get_scores(tokenize(text))
    order = np.argsort(scores)[::-1][:k]
    return [(corpus[i], float(scores[i])) for i in order]

def show_hits(label, hits):
    print(label)
    for rank, (doc, score) in enumerate(hits, 1):
        first_line = doc["text"].splitlines()[0]
        print(f"  {rank}. {doc['id']} | {score:.4f} | {first_line}")

print("BM25 index ready.")


BM25 index ready.


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
embedding_ready = False
embedding_error = None
doc_embeddings = None

if not os.getenv("HF_TOKEN"):
    embedding_error = "HF_TOKEN is not set. Add it to .env after accepting the EmbeddingGemma licence."
    print("SKIPPED:", embedding_error)
else:
    try:
        from sentence_transformers import SentenceTransformer
        embedding_model = SentenceTransformer("google/embeddinggemma-300m", token=os.getenv("HF_TOKEN"))
        doc_embeddings = embedding_model.encode_document(
            [d["text"] for d in corpus], normalize_embeddings=True, show_progress_bar=True
        )
        embedding_ready = True
        print("EmbeddingGemma ready; corpus shape:", doc_embeddings.shape)
    except Exception as exc:
        embedding_error = f"{type(exc).__name__}: {exc}"
        print("SKIPPED: could not load EmbeddingGemma:", embedding_error)

def embedding_retrieve(text, k=3):
    if not embedding_ready:
        raise RuntimeError(embedding_error)
    query_embedding = embedding_model.encode_query([text], normalize_embeddings=True)
    scores = (query_embedding @ doc_embeddings.T)[0]
    order = np.argsort(scores)[::-1][:k]
    return [(corpus[i], float(scores[i])) for i in order]


/home/bryan/css4007-hw3-michaelbryan/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 12744.45it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:  33%|███▎      | 1/3 [00:05<00:10,  5.05s/it]

Batches:  67%|██████▋   | 2/3 [00:07<00:03,  3.47s/it]

Batches: 100%|██████████| 3/3 [00:08<00:00,  2.24s/it]

Batches: 100%|██████████| 3/3 [00:08<00:00,  2.73s/it]

EmbeddingGemma ready; corpus shape: (74, 768)


## My own queries

The relevance labels and expectations below were fixed before running either retriever.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which launchpad was renamed Gagarin's Start? | KZ-11 | **BM25**, because the rare exact string “Gagarin's Start” occurs in the answer chunk. |
| Q-10 | What support lets a low-income undergraduate study without taking a job? | GO-01 | **Embeddings**, because this paraphrases the handbook's purpose rather than copying its main keywords. |


In [5]:
own_queries = [
    {"id": "Q-09", "text": "Which launchpad was renamed Gagarin's Start?", "relevant": ["KZ-11"]},
    {"id": "Q-10", "text": "What support lets a low-income undergraduate study without taking a job?", "relevant": ["GO-01"]},
]
all_queries = fixed_queries + own_queries
print("Queries ready:", ", ".join(q["id"] for q in all_queries))


Queries ready: Q-01, Q-02, Q-03, Q-04, Q-05, Q-06, Q-07, Q-08, Q-09, Q-10


## Results for all ten or more queries: top 3 from each retriever

In [6]:
results_b = []
for item in all_queries:
    print("\n" + "=" * 90)
    print(item["id"], item["text"], "| relevant:", item["relevant"])
    b_hits = bm25_retrieve(item["text"])
    show_hits("BM25 top 3", b_hits)
    b_ids = [d["id"] for d, _ in b_hits]
    b_hit = bool(set(b_ids) & set(item["relevant"]))
    print("  BM25 hit@3:", b_hit)

    e_hits, e_ids, e_hit = [], [], None
    if embedding_ready:
        e_hits = embedding_retrieve(item["text"])
        show_hits("Embedding top 3", e_hits)
        e_ids = [d["id"] for d, _ in e_hits]
        e_hit = bool(set(e_ids) & set(item["relevant"]))
        print("  Embedding hit@3:", e_hit)
    else:
        print("Embedding top 3: SKIPPED —", embedding_error)
    results_b.append({"id": item["id"], "bm25": b_hits, "bm25_hit": b_hit,
                      "embedding": e_hits, "embedding_hit": e_hit})

bm25_total = sum(r["bm25_hit"] for r in results_b)
embedding_total = sum(r["embedding_hit"] for r in results_b if r["embedding_hit"] is not None) if embedding_ready else None
print("\nSUMMARY")
print("Query | BM25 top 3 | hit | Embedding top 3 | hit")
for r in results_b:
    b = ", ".join(d["id"] for d, _ in r["bm25"])
    e = ", ".join(d["id"] for d, _ in r["embedding"]) if embedding_ready else "pending"
    print(f"{r['id']} | {b} | {r['bm25_hit']} | {e} | {r['embedding_hit']}")
print(f"BM25 hit@3 total: {bm25_total}/{len(all_queries)}")
print("Embedding hit@3 total:", f"{embedding_total}/{len(all_queries)}" if embedding_ready else "pending")



Q-01 What is form GR-305 used for? | relevant: ['GO-08', 'GO-12', 'GO-23']
BM25 top 3
  1. GO-08 | 15.8255 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.
  2. GO-12 | 8.1953 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.
  3. GO-06 | 8.0700 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an

Embedding top 3
  1. GO-02 | 0.6652 | An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.
  2. GO-05 | 0.6210 | ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.
  3. GO-17 | 0.6134 | The grant continues only while the student keeps up their studies. If the cumulative GPA falls below 2.67 at the end of the autumn

Embedding top 3
  1. GO-17 | 0.5591 | The grant continues only while the student keeps up their studies. If the cumulative GPA falls below 2.67 at the end of the autumn semester, the February instalment is suspended. It is not reinstated later in the same year, even if the GPA recovers. The October instalment does not have to be returned, unless the student withdraws from the university before 1 December, in which case it must be repaid in full.
  2. GO-18 | 0.4586 | A student who goes on academic leave keeps their grant decision, but payments stop for the months of the leave and resume in the next instalment after they return. A student who transfers to a different programme inside the university keeps the grant. A student who transfers to another university loses it; the grant cannot be moved with them.
  3. GO-15 | 0.4532 | Грант выплачивается двумя равными частями — в октябре и в феврале — на банковский счёт, указанный студентом в форме GR-104. Счёт должен быть открыт на имя самого

## B3 — comparison

| Query | BM25 top 3 | BM25 hit@3 | Embedding top 3 | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, GO-23, GO-22 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | KZ-11, KZ-37, KZ-05 | ✓ | KZ-11, KZ-10, KZ-37 | ✓ |
| Q-10 | GO-01, GO-10, GO-13 | ✓ | GO-01, GO-20, GO-10 | ✓ |
| **hit@3 total** |  | **7/10** |  | **10/10** |

**Both agree:** Q-01 was easy because the rare literal form code `GR-305` appears in the relevant appeal chunks. Both ranked GO-08 and GO-12 first and second; EmbeddingGemma also found the Russian appeal chunk GO-23.

**BM25 did better:** There was no query where BM25 had a hit and embeddings missed—the measured hit@3 totals were 7/10 versus 10/10. Q-09 is the closest mechanism-level example: BM25 directly matched the rare exact string `Gagarin's Start` and put KZ-11 first with 18.2934, while embeddings also put KZ-11 first (0.6315). I therefore cannot honestly claim a BM25 hit@3 win on these ten runs.

**Embeddings did better:** Q-02 is a clear case. The query says “family is poor” and “financial help,” while GO-01 says “low-income household” and “need-based study grant.” BM25 lacked those exact query/document matches and returned unrelated KZ-17 first, whereas the embedding retriever matched the paraphrase and returned GO-01 first. Q-07 and Q-04 show the same advantage: semantic matching found the Russian payment rule GO-15 and the current GPA rule GO-02, respectively.


---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [7]:
from openai import OpenAI
import tiktoken

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""
encoding = tiktoken.get_encoding("o200k_base")
openai_ready = bool(os.getenv("OPENAI_API_KEY"))
client = OpenAI() if openai_ready else None

def context_text(hits):
    return "\n\n".join(f"[{i}] {doc['id']} — {doc['title']}\n{doc['text']}" for i, (doc, _) in enumerate(hits, 1))

def rag_user_prompt(question, hits):
    return f"Context:\n{context_text(hits)}\n\nQuestion: {question}"

def ask(question, hits=None):
    if not openai_ready:
        raise RuntimeError("OPENAI_API_KEY is not set")
    if hits is None:
        messages = [{"role": "user", "content": question}]
    else:
        messages = [{"role": "system", "content": SYSTEM},
                    {"role": "user", "content": rag_user_prompt(question, hits)}]
    response = client.chat.completions.create(model="gpt-5.6-luna", messages=messages)
    return response.choices[0].message.content, response.usage.prompt_tokens, messages

print("OpenAI calls ready." if openai_ready else "SKIPPED: OPENAI_API_KEY is not set.")


OpenAI calls ready.


## C1 — the complete prompt, printed before it is sent

In [8]:
c1_record = None
if embedding_ready and openai_ready:
    chosen = qa[0]
    hits = embedding_retrieve(chosen["question"])
    user_prompt = rag_user_prompt(chosen["question"], hits)
    print("SYSTEM MESSAGE:\n" + SYSTEM)
    print("\nUSER MESSAGE:\n" + user_prompt)
    answer, usage_tokens, messages = ask(chosen["question"], hits)
    chunk_tokens = [len(encoding.encode(d["text"])) for d, _ in hits]
    system_tokens = len(encoding.encode(SYSTEM))
    question_tokens = len(encoding.encode(chosen["question"]))
    context_only = context_text(hits)
    context_tokens = len(encoding.encode(context_only))
    c1_record = {"answer": answer, "usage": usage_tokens, "system": system_tokens,
                 "question": question_tokens, "context": context_tokens,
                 "chunks": chunk_tokens, "chars": len(context_only)}
    print("\nMODEL ANSWER:\n", answer)
    print("\nTOKEN ACCOUNTING:", c1_record)
else:
    print("SKIPPED: C1 needs both a successful EmbeddingGemma load and OPENAI_API_KEY.")


SYSTEM MESSAGE:
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

USER MESSAGE:
Context:
[1] GO-04 — How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] GO-03 — Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although the


MODEL ANSWER:
 A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

TOKEN ACCOUNTING: {'answer': 'A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]', 'usage': 390, 'system': 44, 'question': 18, 'context': 314, 'chunks': [90, 104, 85], 'chars': 1221}


| Part of the input | Tokens (`o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context (including block labels, IDs, and titles) | 314 |
| **Visible-text subtotal** | **376** |
| **`usage.prompt_tokens`** | **390** |

- Number of retrieved chunks: **3**
- Raw text tokens per chunk: **90, 104, 85**
- Formatted context size: **314 tokens / 1,221 characters**

The API count is 14 tokens higher because chat messages are serialised with role and separator formatting that is not present in the three visible strings counted separately.

Retrieval literally adds the selected chunk text—plus my block numbers, chunk IDs, titles, and `Context:` label—to the user message. The model receives this pasted text; it does not open or search `corpus.jsonl` itself.


## C2 — three systems, six questions

In [9]:
c2_results = []
if embedding_ready and openai_ready:
    for item in qa:
        q = item["question"]
        b_hits = bm25_retrieve(q)
        e_hits = embedding_retrieve(q)
        llm_answer, _, _ = ask(q, None)
        bm_answer, _, _ = ask(q, b_hits)
        em_answer, _, _ = ask(q, e_hits)
        row = {"id": item["id"], "question": q, "reference": item["reference"],
               "llm": llm_answer, "bm25": bm_answer, "embedding": em_answer,
               "bm25_hits": b_hits, "embedding_hits": e_hits}
        c2_results.append(row)
        print("\n" + "=" * 90)
        print(item["id"], q)
        print("REFERENCE:", item["reference"])
        print("LLM ONLY:", llm_answer)
        print("BM25 RAG:", bm_answer)
        print("EMBEDDING RAG:", em_answer)
    print("\nManually mark each answer ✓/✗ against the references above; do not infer correctness from string equality.")
else:
    print("SKIPPED: C2 needs both model credentials. The loop is complete and will make 18 calls when rerun.")



C-01 How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
LLM ONLY: Which country or student funding scheme do you mean? “Income band 2” is used by several programs, and the amount and payment date vary by scheme and academic year.
BM25 RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: the first in October and the second in February**. [1]
EMBEDDING RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]



C-02 My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.
LLM ONLY: The appeal deadline and form depend on the **grant program, agency, and country**. Check the refusal/decision letter first—it should state:

- whether you can request a review or appeal;
- the deadline, often counted from the date you received the decision;
- the required form or submission method; and
- where to send it.

If the letter does not specify these, contact the granting agency promptly and ask for its **appeal/reconsideration procedure**. Tell me the **name of the grant program, agency/country, and the date on the refusal letter**, and I can help identify the exact deadline and form.
BM25 RAG: You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**. [3]
EMBEDDING RAG: You have **14 calendar days from the date on the decision letter** to appeal. Us


C-03 What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
LLM ONLY: It depends on the specific need-based grant and institution. Many grants, including the federal Pell Grant, do **not** require a minimum GPA to apply, but you must maintain your school’s **Satisfactory Academic Progress (SAP)** standards—often around a **2.0 GPA**.  

Which grant or school are you referring to?
BM25 RAG: The minimum GPA needed is **2.67 on a 4.0 scale**. [2]
EMBEDDING RAG: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]



C-04 When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
REFERENCE: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.
LLM ONLY: Kazakhstan formally decided to move Kazakh from Cyrillic to a Latin-based alphabet in **October 2017**, by Presidential Decree No. 569 (following President Nursultan Nazarbayev’s announcement in April that year).

The original target was **2025**, but the timetable was extended. The transition is now expected to be completed by **2031**, in stages.
BM25 RAG: Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition was originally due by **2025**, but the target was later postponed; it is now scheduled to finish in **2031**. [1][2]
EMBEDDING RAG: Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition is now due to be completed by **2031**. [1][2]



C-05 When was Narxoz University founded, and what was it called at the start?
REFERENCE: In 1963, as the Almaty Institute of National Economy; it took its current name in 2016.
LLM ONLY: Narxoz University was founded in **1963** as the **Alma-Ata Institute of National Economy**.
BM25 RAG: Narxoz University was founded in 1963 as the Almaty Institute of National Economy. [1]
EMBEDDING RAG: Narxoz University was founded in 1963 as the **Almaty Institute of National Economy**. [1]



C-06 I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
LLM ONLY: The amount depends heavily on your **country, university, program, citizenship/residency, income, assets, and enrollment status**. There is no single master’s-level “need-based grant” amount.

**If you mean the United States:**
- **Federal Pell Grants generally do not cover master’s programs.**
- Graduate students may qualify for:
  - **University need-based grants or scholarships**—amount varies by school and available funds.
  - **Federal Work-Study**, if your school participates.
  - **TEACH Grants**, only for qualifying teacher-preparation programs and service commitments.
  - **State grants**, depending on your state and eligibility.
- Your FAFSA determines your **Student Aid Index (SAI)**, but graduate aid offices usually use their own budget and policies to calculate institutiona

## C2 — three systems, six questions

I marked semantic agreement with the supplied reference. For C-05, the LLM-only answer says “Alma-Ata Institute,” while the reference and chunk say “Almaty Institute,” so I conservatively marked it wrong. For C-06, both RAG answers correctly say that the undergraduate handbook gives no master's amount and points to a separate scheme.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asks which scheme | ✓ 150,000; October and February | ✓ 150,000; October and February |
| C-02 | ✗ gives no deadline/form | ✓ 14 days; GR-305 | ✓ 14 days; GR-305 |
| C-03 | ✗ guesses a generic 2.0 | ✓ 2.67 | ✓ 2.67 |
| C-04 | ✓ October 2017; finish 2031 | ✓ October 2017; finish 2031 | ✓ October 2017; finish 2031 |
| C-05 | ✗ says “Alma-Ata” | ✓ 1963; Almaty Institute | ✓ 1963; Almaty Institute |
| C-06 | ✗ discusses generic US aid | ✓ amount unknown; separate scheme | ✓ not covered here; separate scheme |
| **agrees with reference** | **1/6** | **6/6** | **6/6** |


## C3 — analysis 1: C-03 (minimum GPA)

**Retrieval.** BM25 returned GO-16 (15.7985), GO-02 (14.0587), and GO-06 (13.8516). EmbeddingGemma returned GO-02 (0.6286), GO-20 (0.5691), and archived GO-05 (0.5451). The exact blocks appear in the code output below.

**Effect on the answer.** LLM-only guessed a generic 2.0 standard, but both RAG systems answered 2.67. BM25 cited its block [2] and embeddings cited [1]. Embedding retrieval was more precise at rank 1, although it also brought back the dangerous archived 3.0 rule at rank 3.

**Failure classification.** There was no final-answer failure: GO-02 explicitly says “cumulative GPA is at least 2.67 on the 4.0 scale,” and both generators used it correctly. The archived distractor did not cause a generation failure because its own text clearly says that the 3.0 rule is no longer in force.


In [10]:
item = next(x for x in qa if x["id"] == "C-03")
print(item["id"], item["question"])
print("REFERENCE:", item["reference"])
b_hits = bm25_retrieve(item["question"])
show_hits("BM25", b_hits)
print("\nBM25 CONTEXT SENT:\n" + context_text(b_hits))
if embedding_ready:
    e_hits = embedding_retrieve(item["question"])
    show_hits("\nEmbedding", e_hits)
    print("\nEMBEDDING CONTEXT SENT:\n" + context_text(e_hits))
else:
    print("\nEmbedding output pending HF_TOKEN.")
if not openai_ready:
    print("\nAnswer comparison pending OPENAI_API_KEY.")


C-03 What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
BM25
  1. GO-16 | 15.7985 | The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.
  2. GO-02 | 14.0587 | An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, wh

## C3 — analysis 2: C-06 (master's student)

**Retrieval.** BM25 returned GO-16 (16.9958), GO-01 (14.1725), and GO-20 (13.7202). EmbeddingGemma returned GO-20 (0.6664), GO-16 (0.6161), and GO-01 (0.5539). Both supplied GO-20, but embeddings placed it first.

**Effect on the answer.** LLM-only wandered into US federal aid. BM25 correctly said it did not know the amount and cited GO-20. The embedding answer was slightly worse than expected because it led with “You cannot apply for this need-based grant” instead of literally saying that the master's amount is unknown, but it correctly explained that master's students have a separate scheme and should ask their department.

**Failure classification.** This is not a retrieval failure: both prompts contain GO-20, which says the separate master's scheme “is not described.” Any invented amount would therefore be a generation failure. Neither RAG system invented one; BM25 expressed the required uncertainty most directly.


In [11]:
item = next(x for x in qa if x["id"] == "C-06")
print(item["id"], item["question"])
print("REFERENCE:", item["reference"])
b_hits = bm25_retrieve(item["question"])
show_hits("BM25", b_hits)
print("\nBM25 CONTEXT SENT:\n" + context_text(b_hits))
if embedding_ready:
    e_hits = embedding_retrieve(item["question"])
    show_hits("\nEmbedding", e_hits)
    print("\nEMBEDDING CONTEXT SENT:\n" + context_text(e_hits))
else:
    print("\nEmbedding output pending HF_TOKEN.")
if not openai_ready:
    print("\nAnswer comparison pending OPENAI_API_KEY.")


C-06 I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
BM25
  1. GO-16 | 16.9958 | The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.
  2. GO-01 | 14.1725 | The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on 

## C3 — analysis 3: C-02 (appeal)

**Retrieval.** BM25 returned archived GO-05 (17.2414), GO-08 (14.5287), and GO-12 (13.5735). EmbeddingGemma returned GO-08 (0.6029), GO-12 (0.5591), and GO-11 (0.5343). Thus both prompts contained the two facts, but embeddings avoided the unrelated archived rule.

**Effect on the answer.** LLM-only could only ask for the unknown programme. Both RAG answers became specific and grounded: 14 calendar days from the decision letter, using GR-305. BM25 cited block [3], while embeddings cited block [2].

**Failure classification.** There was no answer failure. GO-12 states “within 14 calendar days ... using form GR-305,” and both generators reproduced it. BM25's irrelevant GO-05 at rank 1 is a retrieval-quality problem, but not a final retrieval failure because GO-12 still appeared within top 3.


In [12]:
item = next(x for x in qa if x["id"] == "C-02")
print(item["id"], item["question"])
print("REFERENCE:", item["reference"])
b_hits = bm25_retrieve(item["question"])
show_hits("BM25", b_hits)
print("\nBM25 CONTEXT SENT:\n" + context_text(b_hits))
if embedding_ready:
    e_hits = embedding_retrieve(item["question"])
    show_hits("\nEmbedding", e_hits)
    print("\nEMBEDDING CONTEXT SENT:\n" + context_text(e_hits))
else:
    print("\nEmbedding output pending HF_TOKEN.")
if not openai_ready:
    print("\nAnswer comparison pending OPENAI_API_KEY.")


C-02 My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.
BM25
  1. GO-05 | 17.2414 | ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.
  2. GO-08 | 14.5287 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.
  3. GO-12 | 13.5735 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the


Embedding
  1. GO-08 | 0.6029 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.
  2. GO-12 | 0.5591 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.
  3. GO-11 | 0.5343 | If a required document is missing, the Grant Office does not refuse the application straight away. The decision is recorded as 'more information needed', and the student is tol

---
# Final questions

**1. Main difference:** BM25 scores lexical overlap using term frequency, inverse document frequency, and length normalisation. Embedding retrieval represents meaning as vectors, so it can match related wording even without the same tokens.

**2. When BM25 can win:** It is strong when an exact rare string matters, such as a form code, number, or proper name. Q-09 deliberately tests the rare phrase “Gagarin's Start.”

**3. When embeddings can win:** They can win on synonyms, paraphrases, and multilingual queries whose surface tokens do not overlap the answer. Q-03 and Q-07 are useful tests in this corpus.

**4. Why cosine similarity:** It compares vector direction (semantic pattern) while removing vector magnitude, giving a normalised score suitable for ranking embeddings.

**5. What RAG sends:** The application sends system instructions, the question, and the text of the retrieved chunks formatted into the prompt. It may also send labels such as chunk numbers, IDs, and titles.

**6. Does the LLM search documents?** No. The retrieval code searches first; the model sees only the text placed in its request.

**7. Irrelevant chunks:** They consume the context budget and can distract or mislead the model. A grounded model may instead correctly say the answer is unavailable.

**8. Why Top-K can help or hurt:** A larger K can recover a missed supporting chunk, but can also add irrelevant or contradictory text (especially the archived rule). The code below tests K=1,3,8 on C-03 and prints the retrieved IDs and model answers.


In [13]:
topk_results = []
item = next(x for x in qa if x["id"] == "C-03")
if embedding_ready:
    for k in (1, 3, 8):
        hits = embedding_retrieve(item["question"], k=k)
        ids = [d["id"] for d, _ in hits]
        if openai_ready:
            answer, _, _ = ask(item["question"], hits)
        else:
            answer = "pending OPENAI_API_KEY"
        topk_results.append((k, ids, answer))
        print(f"K={k} | IDs={ids}\nAnswer: {answer}\n")
else:
    print("SKIPPED: Top-K experiment needs HF_TOKEN (and OPENAI_API_KEY for answers).")


K=1 | IDs=['GO-02']
Answer: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]



K=3 | IDs=['GO-02', 'GO-20', 'GO-05']
Answer: The minimum GPA is **2.67 on a 4.0 scale**. [1]



K=8 | IDs=['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
Answer: The minimum GPA needed to qualify for the 2026 need-based grant is **2.67 on a 4.0 scale**. [1]



**9. Is a vector database required?** No. This notebook stores only 74 embedding vectors in a NumPy array and ranks them with an in-memory matrix multiplication. A vector database becomes useful for scale, persistence, metadata filtering, and operational retrieval, not for the basic RAG idea.

**10. Which retrieval approach worked better?** EmbeddingGemma worked better on retrieval: **10/10 hit@3**, compared with BM25's **7/10**. It recovered paraphrases (Q-02), the current rather than archived GPA rule (Q-04), and the Russian payment rule (Q-07). In C2, however, both BM25 RAG and embedding RAG agreed with the reference on **6/6**, because BM25 still retrieved the necessary evidence for all six questions within its top three.

---
# Conclusion

For this corpus I would build a hybrid retriever that combines EmbeddingGemma's semantic recall with BM25's exact-code matching, then add metadata filtering so archived rules cannot outrank current rules. I would cache document embeddings and keep this evaluation set as a regression test. The experiment shows why both retrieval and generation must be inspected: embeddings won hit@3 (10/10 versus 7/10), but both RAG pipelines reached 6/6 answer agreement because the needed C2 evidence was still present in BM25's context.
